In [0]:
import org.apache.spark.sql.functions._
import com.databricks.dbutils_v1.DBUtilsHolder.dbutils
import org.apache.spark.sql.types._
import org.apache.spark.sql.DataFrame
import scala.collection.mutable.ListBuffer
//import org.apache.spark.sql.types.DataTypes
import org.apache.spark.sql.expressions.UserDefinedFunction
import scala.util.matching.Regex

val dbPath = "dbfs:/FileStore/flight_project.db"
val tablePath = "dbfs:/FileStore/flight" // /flight_clean"
// Recrée la base si nécessaire (le metastore peut avoir disparu même si le dossier existe)
spark.sql(s"""
  CREATE DATABASE IF NOT EXISTS flight_project
  LOCATION '$dbPath'
""")

In [0]:
val flightCleanPath = tablePath + "/flight_clean"
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.flight_clean
  USING DELTA
  LOCATION '$flightCleanPath'
""")

In [0]:

val weatherCleanPath = tablePath + "/weather_clean"
// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.weather_clean
  USING DELTA
  LOCATION '$weatherCleanPath'
""")

In [0]:
// -----------------------------------------------------
// 0) CONFIG GLOBAL
// -----------------------------------------------------
// Active AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
// Définir un seuil de broadcast (ici 10 Mo)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
// Bin size pour range join
spark.conf.set("spark.databricks.optimizer.rangeJoin.binSize", 3600)
// Nombre de partitions de shuffle (à ajuster à la taille de votre cluster)
spark.conf.set("spark.sql.shuffle.partitions", "50")

In [0]:
import org.apache.spark.sql.Column
// --- 1) Lecture brute de weather_clean ---
val weatherRaw = spark.table("flight_project.weather_clean")

// --- 2) Récupérer TOUTES les colonnes sauf AirportId (on garde timestamp !) ---
val rawCols = weatherRaw.columns.filter(_ != "AirportId")

// --- 3) Construire dynamiquement la liste de colonnes à sélectionner :
//      • AirportId -> apt_id
//      • timestamp -> w_ts
//      • toutes les autres laissées telles quelles (nom + type préservés)
// 
//  Seq( col("AirportId").alias("apt_id") ) ++ 
//     rawCols.map {
//       case "timestamp" => col("timestamp").alias("w_ts")
//       case other       => col(other)
//     }
// --------------------------------------------------------------------------
val selectCols: Seq[Column] =
  Seq(col("AirportId").alias("apt_id")) ++
  rawCols.map {
    case "timestamp" => col("timestamp").alias("w_ts")
    case other       => col(other)
  }

// --- 4) Appliquer la sélection et mettre en cache pour réutilisation ---
val weatherSelected = weatherRaw
  .select(selectCols: _*)
  .cache()

// Matérialisation (pour éviter de recharger la table à chaque join)
weatherSelected.count()

In [0]:
// -----------------------------------------------------
// 2) PRÉPARATION des vols avec depart+arrivée
// -----------------------------------------------------
val flightsTS = spark.table("flight_project.flight_clean")
  .withColumn(
    "CRS_ARR_TIMESTAMP",
    expr("timestampadd(MINUTE, CRS_ELAPSED_TIME, CRS_DEP_TIMESTAMP)")
  )
  .repartition(col("FL_DATE")) // partition par date pour la suite

In [0]:
// -----------------------------------------------------
// 3) JOINTURE ORIGINE OPTIMISÉE
// -----------------------------------------------------
// On pose notre hint SUR weatherSelected,
// et on broadcaste weather (car elle est petite)
// pour n’avoir qu’un seul shuffle côté vol.
val weatherOrigHinted = weatherSelected
  .hint("range_join", 3600)

val originCond = 
     flightsTS("ORIGIN_AIRPORT_ID") === weatherOrigHinted("apt_id") &&
     weatherOrigHinted("w_ts") >= flightsTS("CRS_DEP_TIMESTAMP") - expr("INTERVAL 12 HOURS") &&
     weatherOrigHinted("w_ts") <= flightsTS("CRS_DEP_TIMESTAMP")

val withOrigin = flightsTS
  .join(
    broadcast(weatherOrigHinted), // broadcast évite le shuffle météo
    originCond,
    "inner"                       // inner join éligible à range join
  )
  .groupBy(flightsTS.columns.map(col): _*)
  .agg(
    sort_array(collect_list(struct(weatherOrigHinted.columns.map(col): _*)))
      .alias("weather_origin")
  )

In [0]:
// -----------------------------------------------------
// 4) JOINTURE DESTINATION OPTIMISÉE
// -----------------------------------------------------
val weatherDestSelected = weatherSelected
  .withColumnRenamed("apt_id", "dest_apt")
  .withColumnRenamed("w_ts",    "w_ts_dest")
val weatherDestHinted = weatherDestSelected
  .hint("range_join", 3600)

val destCond = 
     withOrigin("DEST_AIRPORT_ID") === weatherDestHinted("dest_apt") &&
     weatherDestHinted("w_ts_dest") >= withOrigin("CRS_ARR_TIMESTAMP") - expr("INTERVAL 12 HOURS") &&
     weatherDestHinted("w_ts_dest") <= withOrigin("CRS_ARR_TIMESTAMP")

val finalJoined = withOrigin
  .join(
    broadcast(weatherDestHinted),
    destCond,
    "inner"
  )
  .groupBy(withOrigin.columns.map(col): _*)
  .agg(
    sort_array(collect_list(struct(weatherDestHinted.columns.map(col): _*)))
      .alias("weather_dest")
  )

In [0]:
// -----------------------------------------------------
// 5) VÉRIFIER LE PLAN
// -----------------------------------------------------
finalJoined.explain("extended")
// On doit voir :
//   • BroadcastHashJoin côté météo
//   • RangeJoin (binning) appliqué
//   • Adaptive Query Execution
//   • shuffle partitions = 50


In [0]:
val joinCleanPath = tablePath + "/joined_data"
dbutils.fs.rm(joinCleanPath, recurse = true)

// Vérifie si le dossier Delta existe dans DBFS
val deltaJoinCleanExists = dbutils.fs.ls(tablePath)
  .exists(_.path.stripSuffix("/") == joinCleanPath)


if (!deltaJoinCleanExists) {

  println("Création de la table delta")
  val dates = finalJoined.select("FL_DATE").distinct().as[java.sql.Date].collect()
  dates.foreach { date =>
    finalJoined
      .filter(col("FL_DATE") === lit(date))
      .coalesce(1)
      .write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .save(joinCleanPath)
  }  
  }
  


In [0]:

// Création de la table Delta externe (persistante)
spark.sql(s"""
  CREATE TABLE IF NOT EXISTS flight_project.joined_data
  USING DELTA
  LOCATION '$joinCleanPath'
""")

In [0]:
%sql
select count(*)
from flight_project.joined_data
--limit 10

In [0]:
%sql
describe flight_project.joined_data

In [0]:
%sql
select *
from flight_project.joined_data
limit 10

In [0]:
// 1) Relire la table Delta “propre”, sans aucun hint
val joinedDF = spark.table("flight_project.joined_data")

// 2) Calculer dynamiquement le nombre de lags
val lagCount = joinedDF
  .select(max(size(col("weather_origin"))).alias("maxLag"))
  .as[Int]
  .first()

println(s"Nombre maximal de pas horaires (lags) détecté = $lagCount")


In [0]:
// 3) Récupérer une seule fois le schéma des StructFields
val originStruct = joinedDF
  .schema("weather_origin").dataType
  .asInstanceOf[ArrayType]
  .elementType
  .asInstanceOf[StructType]

val fieldSchemas = originStruct.fields

// 4) Générer dynamiquement les colonnes « wide » pour origin & dest
val originCols = for {
  lag <- 0 until lagCount
  f   <- fieldSchemas
} yield col("weather_origin")
         .getItem(lag)
         .getField(f.name)
         .alias(s"orig_${f.name}_lag${lag}")

val destCols = for {
  lag <- 0 until lagCount
  f   <- fieldSchemas
} yield col("weather_dest")
         .getItem(lag)
         .getField(f.name)
         .alias(s"dest_${f.name}_lag${lag}")



In [0]:
// 5) Construire le DataFrame aplati
val flatDF = joinedDF.select(
  // on garde toutes les colonnes vol/étiquette
  joinedDF.columns.filterNot(c => c == "weather_origin" || c == "weather_dest")
    .map(col) ++
  originCols ++
  destCols: _*
)